# Lesson 9 — Tests and model serving
November 2, 2026 · UCB MFE

Run setup before class. Demo A accompanies slide 14; Demo B accompanies slide 26.
The model uses synthetic data solely to illustrate serving mechanics.

## Setup — run before class
Start Jupyter with `cd demo` then `uv run --locked jupyter lab ../Lesson_9_Demos.ipynb`.
This cell starts the API in a separate process on this computer. Leave it running for Demo B.

In [ ]:
from pathlib import Path
import os
import sys

root = Path.cwd()
demo = root if (root / "app.py").exists() else root / "demo"
if not (demo / "app.py").exists():
    raise RuntimeError("Open this notebook from the Lecture 9 folder or its demo folder")
os.chdir(demo)
sys.path.insert(0, str(demo))

import httpx
import joblib
import pytest
from demo_helpers import run_fee_test, start_server, stop_server

server, base_url = start_server()

## Demo A — a function can run and still be wrong
Two shares at $100, with a fee of 10 basis points. What should the fee be?
One basis point is 0.01%; 10 basis points is 0.1%.

In [ ]:
from fees_buggy import trading_fee as buggy_fee
print("Fee returned: $", buggy_fee(shares=2, price=100, fee_bps=10), sep="")

### Run the independent check
Read the failure: what did the calculation return, and what did the test expect?
The test below deliberately fails; the helper displays the output without interrupting the notebook.

In [ ]:
run_fee_test("buggy")

### Inspect the correction, then run the same check
The denominator changes from 100 to 10,000. The expected result remains $0.20.

In [ ]:
import inspect
from fees import trading_fee
print(inspect.getsource(trading_fee))
run_fee_test("fixed")

### Boundary question
What should happen with zero shares? With a negative price?

In [ ]:
print("Zero shares:", trading_fee(0, 100, 10))
try:
    trading_fee(2, -100, 10)
except ValueError as error:
    print("Negative price rejected:", error)

## Demo B — the same model, called through an API
Both features are fractions. `momentum_20d=0.02` means a 2% trailing 20-day return.
`volatility_20d=0.15` means 15% annualized volatility estimated from 20 days.

First call the saved pipeline directly. It contains both the fitted scaler and model.

In [ ]:
bundle = joblib.load("artifacts/model.joblib")
payload = {"momentum_20d": 0.02, "volatility_20d": 0.15}
row = [[payload["momentum_20d"], payload["volatility_20d"]]]
direct_prediction = float(bundle["pipeline"].predict(row)[0])
print("Direct prediction:", round(direct_prediction, 6))

### Send the same input over HTTP
The notebook is the client; the separately running Python process is the server.

In [ ]:
response = httpx.post(base_url + "/predict", json=payload, timeout=5, trust_env=False)
print("Request:", payload)
print("Status:", response.status_code)
print("Response:", response.json())
print("Request ID:", response.headers.get("X-Request-ID"))
response.raise_for_status()
assert response.json()["prediction"] == pytest.approx(direct_prediction)
print("The direct and API predictions agree.")

### Omit a required input
Predict the status before running. Does the request reach the model?

In [ ]:
invalid = httpx.post(base_url + "/predict", json={"momentum_20d": 0.02}, timeout=5, trust_env=False)
print("Status:", invalid.status_code)
print("Validation:", invalid.json()["detail"][0]["loc"], invalid.json()["detail"][0]["msg"])
assert invalid.status_code == 422

### Run the service checks
The tests cover valid output, invalid input, inference failure, and saved preprocessing.

In [ ]:
import subprocess
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_app.py"], capture_output=True, text=True)
print(result.stdout)
assert result.returncode == 0, result.stderr

## After class — stop the local server
Run this before restarting the notebook kernel or closing Jupyter.

In [ ]:
stop_server(server)

## Optional Docker extension — slide 36
Before class, build the image from the demo folder: `docker build -t mfe-lesson9:classroom-v1 .`

In a terminal, run `docker run --rm --name mfe-lesson9-demo -p 127.0.0.1:8770:8000 mfe-lesson9:classroom-v1`.
Then repeat the valid request with `base_url = "http://127.0.0.1:8770"`.
The host port is 8770; the service listens on port 8000 inside the container.
Stop with Ctrl-C. Restart the same image and repeat the request: the packaged model remains available.